# HONESTAI: Measuring Metacognitive Abilities in LLMs
This notebook loads the HONESTAI dataset, runs the evaluator against simulated predictions, and produces calibration and hallucination visualizations.

In [ ]:
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")
random.seed(42)
np.random.seed(42)

In [ ]:
with open('honestai_dataset.json','r',encoding='utf-8') as f:
    dataset = json.load(f)
df = pd.DataFrame(dataset)
df.head()

In [ ]:
# Simulate simple model predictions
def simulate_model(dataset, overconfident=False, hallucinate_pct=0.2):
    preds = []
    for item in dataset:
        tid = item['id']
        ttype = item['task_type']
        if ttype == 'impossible':
            if random.random() < hallucinate_pct:
                pred = 'invented answer'
                conf = 0.8 if overconfident else 0.6
                abstain = False
            else:
                pred = 'UNANSWERABLE'
                conf = 0.0
                abstain = True
        elif ttype == 'fact':
            if random.random() < 0.85:
                pred = item['ground_truth']
                conf = 0.9 if not overconfident else 0.98
                abstain = False
            else:
                pred = 'wrong'
                conf = 0.9 if overconfident else 0.4
                abstain = False
        elif ttype == 'trick':
            if random.random() < 0.7:
                pred = item['ground_truth']
                conf = 0.85 if not overconfident else 0.97
                abstain = False
            else:
                pred = 'wrong'
                conf = 0.85 if overconfident else 0.35
                abstain = False
        else:
            if random.random() < 0.8:
                pred = item['ground_truth']
                conf = 0.75 if not overconfident else 0.95
                abstain = False
            else:
                pred = 'incorrect' if item['ground_truth'] == 'correct' else 'correct'
                conf = 0.6 if overconfident else 0.3
                abstain = False
        preds.append({'id': tid, 'prediction': str(pred), 'confidence': float(conf), 'abstain': bool(abstain)})
    return preds

preds = simulate_model(dataset, overconfident=False, hallucinate_pct=0.15)
with open('predictions_simulated.jsonl','w',encoding='utf-8') as f:
    for p in preds:
        f.write(json.dumps(p, ensure_ascii=False) + '\n')

In [ ]:
# Evaluate using evaluator.py
import evaluator as ev
results = ev.evaluate(dataset, preds)
ev.pretty_print(results)

In [ ]:
# Calibration / Reliability diagram
import matplotlib.pyplot as plt
import numpy as np
def reliability_diagram(preds, dataset, n_bins=10, title="Reliability Diagram"):
    pred_map = {p['id']: p for p in preds}
    confidences = []
    corrects = []
    for item in dataset:
        p = pred_map.get(item['id'], {'prediction':'UNANSWERABLE','confidence':0.0,'abstain':True})
        if p.get('abstain', False):
            continue
        conf = p.get('confidence', 0.0)
        pred = p.get('prediction','')
        corr = 1.0 if ev.compare_answers(item['ground_truth'], pred) else 0.0
        confidences.append(conf)
        corrects.append(corr)
    if not confidences:
        print('No non-abstained predictions to plot')
        return
    df = pd.DataFrame({'conf':confidences,'corr':corrects})
    df['bin'] = pd.cut(df['conf'], bins=np.linspace(0,1,n_bins+1), include_lowest=True)
    bin_stats = df.groupby('bin').agg(avg_conf=('conf','mean'), acc=('corr','mean'), count=('corr','count')).dropna()
    plt.figure(figsize=(6,5))
    plt.plot(bin_stats['avg_conf'], bin_stats['acc'], marker='o')
    plt.plot([0,1],[0,1], linestyle='--', color='gray')
    plt.xlabel('Average confidence')
    plt.ylabel('Accuracy')
    plt.title(title)
    plt.ylim(-0.05,1.05)
    plt.xlim(-0.05,1.05)
    plt.show()

reliability_diagram(preds, dataset, title="Simulated model reliability")